# Lab 3 · Build **Silver** — clean & conform

Turn raw bronze into typed, deduplicated **silver** fact tables. Along the way we use core DE features: **cast/clean**, **upsert (MERGE)**, **schema evolution**, **Change Data Feed**, **time travel**.

> **Attach** the `lh_resident360` Lakehouse first.

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

## 1. fact_meal_log — drop invalid dates, cast calories
Bad dates (e.g. `31/06/2026`) become null and are filtered; blank calories stay null.

In [ ]:
meal = (spark.table("bronze.h365_meal_logs")
        .withColumn("log_date", F.to_date("log_date"))
        .filter(F.col("log_date").isNotNull())
        .withColumn("calories", F.col("calories").cast("int"))
        .withColumn("healthier_choice_flag", F.upper(F.trim("healthier_choice_flag"))))
meal.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_meal_log")
print("silver.fact_meal_log:", spark.table("silver.fact_meal_log").count(), "rows")

## 2. fact_event_attendance — cast date, enable **Change Data Feed**

In [ ]:
ev = (spark.table("bronze.h365_event_bookings")
      .withColumn("event_date", F.to_date("event_date")))
ev.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_event_attendance")
spark.sql("ALTER TABLE silver.fact_event_attendance SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("silver.fact_event_attendance:", spark.table("silver.fact_event_attendance").count(), "rows")

## 3. fact_programme_enrolment

In [ ]:
pr = (spark.table("bronze.h365_programme_enrolments")
      .withColumn("enrol_date", F.to_date("enrol_date")))
pr.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_programme_enrolment")
print("silver.fact_programme_enrolment:", spark.table("silver.fact_programme_enrolment").count(), "rows")

## 4. fact_rewards — Healthpoints ledger

In [ ]:
rw = (spark.table("bronze.h365_rewards")
      .withColumn("txn_date", F.to_date("txn_date"))
      .withColumn("points_earned", F.col("points_earned").cast("int"))
      .withColumn("points_redeemed", F.col("points_redeemed").cast("int")))
rw.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_rewards")
print("silver.fact_rewards:", spark.table("silver.fact_rewards").count(), "rows")

## 5. fact_evoucher_redemption

In [ ]:
vr = (spark.table("bronze.h365_evoucher_redemptions")
      .withColumn("redeemed_date", F.to_date("redeemed_date"))
      .withColumn("voucher_value_sgd", F.col("voucher_value_sgd").cast("double"))
      .withColumn("healthpoints_spent", F.col("healthpoints_spent").cast("int")))
vr.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_evoucher_redemption")
print("silver.fact_evoucher_redemption:", spark.table("silver.fact_evoucher_redemption").count(), "rows")

## 6. fact_challenge — challenge participation & progress

In [ ]:
ch = (spark.table("bronze.h365_challenges")
      .withColumn("enrol_date", F.to_date("enrol_date"))
      .withColumn("progress_pct", F.col("progress_pct").cast("double")))
ch.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_challenge")
print("silver.fact_challenge:", spark.table("silver.fact_challenge").count(), "rows")

## 7. DE feature demo — schema evolution + time travel
Append one row that adds a **new column** (`logged_via`) with `mergeSchema`, then read history.

In [ ]:
extra = spark.sql("""SELECT
    string('RESIDENT_00001') AS resident_id, date('2026-06-30') AS log_date,
    'Snack' AS meal_type, 'Apple' AS food_item, 52 AS calories,
    'Y' AS healthier_choice_flag, 'App' AS logged_via""")
extra.write.mode("append").option("mergeSchema", True).saveAsTable("silver.fact_meal_log")
display(spark.sql("DESCRIBE HISTORY silver.fact_meal_log").select("version", "timestamp", "operation"))

## 8. Verify — six silver tables

In [ ]:
for t in ["fact_meal_log","fact_event_attendance","fact_programme_enrolment",
          "fact_rewards","fact_evoucher_redemption","fact_challenge"]:
    print(f"silver.{t:26s}", spark.table(f"silver.{t}").count(), "rows")